# Notebook 3c – Inference: Augmented Development Set (Spanish)

Generate simplifications for all 76 segments of the augmented development
set using all four prompt configurations and save the results as CSV files for evaluation.

**Note:** Unlike the test set inference notebooks, no block splitting is needed here
since the augmented set is small enough to be processed in a single run
within the Groq free-tier daily token limit.

---

### Requirements

A valid Groq API key is required. Set it as an environment variable before running:
```bash
export GROQ_API_KEY=your_key_here
```

### Expected data directory structure

```
data/
└── augmented/
    └── es_augmented_dataset.csv   ← output of Notebook 2
```

## Installation

In [ ]:
!pip install groq pandas tqdm

## 1. Imports and setup

In [ ]:
import os
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise ValueError("Set the GROQ_API_KEY environment variable before running.")

MODEL      = "llama-3.3-70b-versatile"
TEMP       = 0.3
MAX_TOKENS = 200
DELAY      = 1.0

N_SAMPLE     = None
DATASET_PATH = 'data/augmented/es_augmented_dataset.csv'
OUTPUT_DIR   = 'predictions_dev/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

client = Groq(api_key=GROQ_API_KEY)
print(f'Model:   {MODEL}')
print(f'Dataset: {DATASET_PATH}')
print(f'Output:  {OUTPUT_DIR}')

## 2. Load augmented development dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)

df = df.dropna(subset=['original_text', 'simplified_text']).copy()
df['original_text']   = df['original_text'].astype(str).str.strip()
df['simplified_text'] = df['simplified_text'].astype(str).str.strip()
df = df[(df['original_text'].str.len() > 0) & (df['simplified_text'].str.len() > 0)]

df['sent_id'] = df['document_id'].astype(str) + '_' + df['original_sentence_id'].astype(str)

if N_SAMPLE is not None:
    df = df.head(N_SAMPLE).copy()
    print(f'⚠  Test mode: {N_SAMPLE} segments')

print(f'Segments loaded: {len(df)}')
if 'origin' in df.columns:
    print('Distribution by origin:')
    print(df['origin'].value_counts().to_string())
df.head(3)

## 3. Prompt definitions

The four prompt configurations are defined below.

> To run only a subset of configurations, remove entries from the `PROMPTS` dictionary
> at the bottom of this cell.

In [ ]:
PROMPT_SZS = """Eres un experto en simplificación de textos en español para personas con dificultades de comprensión lectora, como personas con discapacidad intelectual o bajo nivel de alfabetización. Reescribe el siguiente texto para que sea más fácil de entender. Usa frases simples, claras y directas en voz activa. No añadas, elimines ni cambies información importante.

Texto a simplificar:
{texto}

Texto simplificado:
"""

PROMPT_AZS = """Eres un experto en simplificación de textos en español para personas con dificultades de comprensión lectora, como personas con discapacidad intelectual, bajo nivel de alfabetización o conocimiento limitado del idioma. Tu objetivo es transformar textos complejos en versiones claras, sencillas y accesibles, preservando siempre el significado original completo.

REGLA ABSOLUTA: Tu única tarea es reescribir el texto de entrada con palabras más simples. NUNCA añadas contenido que no esté literalmente en el texto de entrada. Si el texto es una pregunta, simplifica la pregunta, no la respondas.

Sigue estas instrucciones:
- Usa frases cortas con una sola idea cada una. Separa las ideas en oraciones distintas. No fragmentes en exceso: si dos ideas están relacionadas, puedes mantenerlas en una sola frase.
- Elimina los incisos y aposiciones. Conviértelas en oraciones independientes que vayan justo después.
- Usa el presente de indicativo y la voz activa. Evita la voz pasiva, los gerundios, el subjuntivo y los tiempos compuestos. Evita también la pasiva refleja (se vende, se garantiza).
- Respeta el tiempo verbal del texto original cuando narre hechos pasados.
- Elimina los conectores complejos. Sustituye "por lo tanto" y "por consiguiente" por "por eso"; "sin embargo" y "no obstante" por "pero" o "aunque".
- Usa palabras sencillas y de uso frecuente. Sustituye los términos técnicos, abstractos o poco comunes.
- NUNCA expliques palabras cotidianas ni organismos conocidos. Solo explica siglas y tecnicismos legales poco conocidos en una oración independiente justo después.
- Evita la doble negación.
- Transforma las construcciones nominales en oraciones con verbo activo.
- Cuando enumeres tres o más elementos, usa lista con guiones (–).
- Conserva toda la información del texto original.
- Entrega únicamente el texto simplificado. No escribas notas ni ofrezcas alternativas.

RECUERDA: NUNCA añadas contenido que no esté en el texto de entrada.

Texto a simplificar:
{texto}

Texto simplificado:

"""

PROMPT_SFS = """Eres un experto en simplificación de textos en español para personas con dificultades de comprensión lectora, como personas con discapacidad intelectual o bajo nivel de alfabetización. Reescribe el siguiente texto para que sea más fácil de entender. Usa frases simples, claras y directas en voz activa. No añadas, elimines ni cambies información importante.

--- EJEMPLO 1 ---
Original: El impuesto es una cuota involuntaria que deben pagar al Estado los individuos mayores, las empresas y las organizaciones, por diferentes medios de pago, según las posibilidades del contribuyente.
Simplificado: El impuesto es una cuota obligatoria que le pagan al Estado los individuos mayores de edad, las empresas y las organizaciones. Estos impuestos se cancelan por diferentes medios de pago, según las posibilidades del contribuyente.

--- EJEMPLO 2 ---
Original: Así, la Organización Internacional del Trabajo (OIT), agencia de la ONU, la declara como parte de los derechos humanos y señala: La previsión social constituye un sistema de reglas e instituciones para hacer efectiva la protección a la que alude la seguridad social que, a su vez, está encaminada a cubrir aspectos esenciales de la vida de las personas.
Simplificado: Así, la Organización Internacional del Trabajo, agencia de la Organización de las Naciones Unidas, la declara parte de los derechos humanos. Esta señala que: La previsión social constituye un sistema de reglas e instituciones. Pretende hacer efectiva la protección que busca la seguridad social y está encaminada a cubrir las necesidades de las personas.

--- EJEMPLO 3 ---
Original: Entre las instituciones del Estado chileno que cautelan las relaciones de personas, empresas y Estado, con ajuste a las normas legales y reglamentarias en materias económicas y financieras, uso de recursos públicos y acciones de particulares, las más relevantes son: legisla y ejerce control en instituciones del Estado.
Simplificado: Las instituciones del Estado chileno vigilan las relaciones entre personas, empresas y Estado. Buscan ajustar las normas legales y reglamentarias en materia económica y financiera. Además, regulan el uso de recursos públicos y acciones de particulares. Las más relevantes son: legisla y ejerce control en instituciones del Estado.

Ahora simplifica el siguiente texto. Conserva toda la información. No añadas nada nuevo.

Texto a simplificar:
{texto}

Texto simplificado:
"""

PROMPT_AFS = """Eres un experto en simplificación de textos en español para personas con dificultades de comprensión lectora, como personas con discapacidad intelectual, bajo nivel de alfabetización o conocimiento limitado del idioma. Tu objetivo es transformar textos complejos en versiones claras, sencillas y accesibles, preservando siempre el significado original completo.

REGLA ABSOLUTA: Tu única tarea es reescribir el texto de entrada con palabras más simples. NUNCA añadas contenido que no esté literalmente en el texto de entrada. Si el texto es una pregunta, simplifica la pregunta, no la respondas.

Sigue estas instrucciones:
- Usa frases cortas con una sola idea cada una. Separa las ideas en oraciones distintas. No fragmentes en exceso: si dos ideas están relacionadas, puedes mantenerlas en una sola frase.
- Elimina los incisos y aposiciones. Conviértelas en oraciones independientes que vayan justo después.
- Usa el presente de indicativo y la voz activa. Evita la voz pasiva, los gerundios, el subjuntivo y los tiempos compuestos. Evita también la pasiva refleja (se vende, se garantiza).
- Respeta el tiempo verbal del texto original cuando narre hechos pasados.
- Elimina los conectores complejos. Sustituye "por lo tanto" y "por consiguiente" por "por eso"; "sin embargo" y "no obstante" por "pero" o "aunque".
- Usa palabras sencillas y de uso frecuente. Sustituye los términos técnicos, abstractos o poco comunes.
- NUNCA expliques palabras cotidianas ni organismos conocidos. Solo explica siglas y tecnicismos legales poco conocidos en una oración independiente justo después.
- Evita la doble negación.
- Transforma las construcciones nominales en oraciones con verbo activo.
- Cuando enumeres tres o más elementos, usa lista con guiones (–).
- Conserva toda la información del texto original.
- Entrega únicamente el texto simplificado. No escribas notas ni ofrezcas alternativas.

RECUERDA: NUNCA añadas contenido que no esté en el texto de entrada.

A continuación tienes tres ejemplos del estilo de simplificación esperado:

--- EJEMPLO 1 ---
Original: El impuesto es una cuota involuntaria que deben pagar al Estado los individuos mayores, las empresas y las organizaciones, por diferentes medios de pago, según las posibilidades del contribuyente.
Simplificado: El impuesto es una cuota obligatoria que le pagan al Estado los individuos mayores de edad, las empresas y las organizaciones. Estos impuestos se cancelan por diferentes medios de pago, según las posibilidades del contribuyente.

--- EJEMPLO 2 ---
Original: Así, la Organización Internacional del Trabajo (OIT), agencia de la ONU, la declara como parte de los derechos humanos y señala: La previsión social constituye un sistema de reglas e instituciones para hacer efectiva la protección a la que alude la seguridad social que, a su vez, está encaminada a cubrir aspectos esenciales de la vida de las personas.
Simplificado: Así, la Organización Internacional del Trabajo, agencia de la Organización de las Naciones Unidas, la declara parte de los derechos humanos. Esta señala que: La previsión social constituye un sistema de reglas e instituciones. Pretende hacer efectiva la protección que busca la seguridad social y está encaminada a cubrir las necesidades de las personas.

--- EJEMPLO 3 ---
Original: Entre las instituciones del Estado chileno que cautelan las relaciones de personas, empresas y Estado, con ajuste a las normas legales y reglamentarias en materias económicas y financieras, uso de recursos públicos y acciones de particulares, las más relevantes son: legisla y ejerce control en instituciones del Estado.
Simplificado: Las instituciones del Estado chileno vigilan las relaciones entre personas, empresas y Estado. Buscan ajustar las normas legales y reglamentarias en materia económica y financiera. Además, regulan el uso de recursos públicos y acciones de particulares. Las más relevantes son: legisla y ejerce control en instituciones del Estado.

Ahora simplifica el siguiente texto aplicando las mismas reglas y el mismo estilo de los ejemplos.

Texto a simplificar:
{texto}

Texto simplificado:
"""

PROMPTS = {
    'SZS': PROMPT_SZS,
    'AZS': PROMPT_AZS,
    'SFS': PROMPT_SFS,
    'AFS': PROMPT_AFS,
}

print('Prompts defined:')
for name, p in PROMPTS.items():
    print(f'  {name}: ~{len(p.split())} words in prompt')

## 4. Simplification function

In [ ]:
MARKER = 'Texto simplificado:'

def extract_simplification(response: str) -> str:
    if MARKER in response:
        return response.split(MARKER)[-1].strip()
    return response.strip()


def simplify(text: str, prompt_template: str,
             model: str = MODEL, max_retries: int = 3) -> str:
    full_prompt = prompt_template.format(texto=text)
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=TEMP,
                max_tokens=MAX_TOKENS,
                messages=[{'role': 'user', 'content': full_prompt}]
            )
            return extract_simplification(resp.choices[0].message.content)
        except Exception as e:
            print(f'    [Attempt {attempt}/{max_retries}] Error: {e}')
            time.sleep(DELAY * attempt * 2)
    return '[ERROR]'

print('simplify() defined.')

## 5. Inference all prompt configurations

In [ ]:
results = {}

for prompt_name, template in PROMPTS.items():
    print(f'\n{"="*65}')
    print(f'PROMPT: {prompt_name}  |  Model: {MODEL}')
    print(f'{"="*65}')

    predictions = []
    errors = 0

    for i, row in tqdm(df.iterrows(), total=len(df), desc=prompt_name):
        pred = simplify(row['original_text'], template)
        if pred == '[ERROR]':
            errors += 1
        predictions.append(pred)
        time.sleep(DELAY)

    df_result = df[['sent_id', 'original_text', 'simplified_text']].copy()
    df_result = df_result.rename(columns={
        'original_text':   'complex',
        'simplified_text': 'simple',
    })
    df_result['simplified'] = predictions
    df_result['prompt']     = prompt_name

    df_clean = df_result[df_result['simplified'] != '[ERROR]'].copy()

    output_path = f'{OUTPUT_DIR}predictions_{prompt_name}_dev_es.csv'
    df_clean.to_csv(output_path, index=False)
    results[prompt_name] = df_clean

    print(f'  Segments processed: {len(df)}')
    print(f'  Errors:             {errors}')
    print(f'  Saved to:           {output_path}')

print(f'\n✓ Inference completed for {len(PROMPTS)} configurations.')

## 6. Visual inspection — compare predictions against the human reference

In [ ]:
idx_example = df[df['original_text'].str.len() > 100].index[0]
sent_example = df.loc[idx_example, 'sent_id']

print('=' * 70)
print(f'Segment: {sent_example}')
print('=' * 70)
print()
print('ORIGINAL:')
print(df.loc[idx_example, 'original_text'])
print()
print('HUMAN REFERENCE:')
print(df.loc[idx_example, 'simplified_text'])
print()

for prompt_name, df_res in results.items():
    row = df_res[df_res['sent_id'] == sent_example]
    if len(row) > 0:
        print(f'--- {prompt_name} ---')
        print(row.iloc[0]['simplified'])
        print()